# Solutions: Notebook 09 — Quantile Treatment Effects

Complete solutions for all exercises in Notebook 09.

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.linalg import lstsq

warnings.filterwarnings("ignore")

from panelbox.models.quantile import PooledQuantile
from panelbox.models.quantile.treatment_effects import QuantileTreatmentEffects

plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 11
np.random.seed(42)

BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"

data = pd.read_csv(DATA_DIR / "labor_program.csv")
print(f"Data loaded: {data.shape}")
print(data.head())

## Exercise 1: CQTE vs UQTE (Easy)

In [ ]:
# SOLUTION: Exercise 1 — CQTE vs UQTE

# 1. Difference between CQTE and UQTE:
print("1. DIFFERENCE BETWEEN CQTE AND UQTE")
print("=" * 60)
print("""
CONDITIONAL QTE (CQTE):
  - Measures treatment effect at the τ-quantile of Y,
    CONDITIONAL on covariates X
  - Answers: "Among workers with the same education/age,
    how does the program affect low vs high earners?"
  - Estimated via: Q_τ(Y|D,X) = X'β(τ) + δ(τ)·D
  - δ(τ) is the CQTE

UNCONDITIONAL QTE (UQTE):
  - Measures treatment effect on the τ-quantile of the
    POPULATION distribution of Y
  - Answers: "How does the program shift the 10th percentile
    of the overall earnings distribution?"
  - Estimated via RIF regression (Firpo et al. 2009)
""")

# 2. Example where they differ:
print("2. EXAMPLE WHERE CQTE ≠ UQTE")
print("=" * 60)
print("""
Consider a training program where:
- Effect on women: +$500 at all quantiles (conditional)
- Effect on men: +$500 at all quantiles (conditional)
- But women earn less on average

CQTE would show δ(τ) = $500 for all τ (constant)
UQTE would show:
  - UQTE(0.1) > $500 (lower quantiles are mostly women,
    who all get same TE but this shifts more of them up)
  - UQTE(0.9) < $500 (upper quantiles are mostly men)

→ The composition effect makes UQTE vary even when CQTE is constant!
""")

# 3. Policy relevance:
print("3. WHICH IS MORE RELEVANT FOR POLICY?")
print("=" * 60)
print("""
UQTE is typically more relevant for policy because:
- Policy makers care about POPULATION outcomes (overall inequality)
- CQTE conditions on X, which may not be policy-relevant groupings
- UQTE directly answers: "How does this program change inequality?"
- UQTE maps to welfare measures (Gini, poverty rate) more directly

However, CQTE is useful for:
- Understanding mechanisms (WHY effects vary)
- Targeting: if we can condition on X, who benefits most?
- Internal validity: easier to interpret causally with controls
""")

# Numerical comparison
print("\n4. NUMERICAL COMPARISON ON OUR DATA")
print("=" * 60)

tau_grid = [0.1, 0.25, 0.5, 0.75, 0.9]

# CQTE via PooledQuantile
y = data["earnings"].values
X = np.column_stack(
    [
        np.ones(len(data)),
        data["treatment"].values,
        data["education"].values,
        data["age"].values,
        data["experience"].values,
    ]
)

cqte_vals = {}
for tau in tau_grid:
    try:
        model = PooledQuantile(y, X, entity_id=data["id"].values, quantiles=tau)
        result = model.fit(se_type="cluster")
        cqte_vals[tau] = result.params.ravel()[1]
    except:
        cqte_vals[tau] = np.nan

# UQTE via RIF
qte_model = QuantileTreatmentEffects(
    data=data,
    outcome="earnings",
    treatment="treatment",
    covariates=["education", "age", "experience"],
    entity_col="id",
    time_col="period",
)

try:
    rif_result = qte_model.estimate_qte(tau=np.array(tau_grid), method="unconditional")
    uqte_vals = {tau: rif_result.qte_results[tau]["qte"] for tau in tau_grid}
except:
    uqte_vals = dict.fromkeys(tau_grid, np.nan)

print(f"{'\u03c4':<6} {'CQTE':>10} {'UQTE':>10} {'Difference':>12}")
print("-" * 40)
for tau in tau_grid:
    c = cqte_vals.get(tau, np.nan)
    u = uqte_vals.get(tau, np.nan)
    d = c - u if not (np.isnan(c) or np.isnan(u)) else np.nan
    print(f"{tau:<6.2f} {c:>10.1f} {u:>10.1f} {d:>12.1f}")

## Exercise 2: Manual RIF Regression (Easy)

In [ ]:
# SOLUTION: Exercise 2 — Manual RIF Regression

tau_ex2 = 0.5
y_all = data["earnings"].values
d_all = data["treatment"].values
n = len(y_all)

# Step 1: Compute sample quantile
Q_tau = np.quantile(y_all, tau_ex2)
print(f"Step 1: Sample median Q_{tau_ex2} = ${Q_tau:.2f}")

# Step 2: Estimate density at quantile using Gaussian kernel
bandwidth = 1.06 * np.std(y_all) * n ** (-1 / 5)
print(f"\nStep 2: Bandwidth (Silverman's rule) = {bandwidth:.2f}")

kernel_values = np.exp(-0.5 * ((y_all - Q_tau) / bandwidth) ** 2)
f_tau = np.mean(kernel_values) / (bandwidth * np.sqrt(2 * np.pi))
print(f"Density at Q_{tau_ex2}: f(Q_\u03c4) = {f_tau:.6f}")

# Step 3: Compute RIF for each observation
indicator = (y_all <= Q_tau).astype(float)
RIF = Q_tau + (tau_ex2 - indicator) / f_tau

print("\nStep 3: RIF statistics:")
print(f"  Mean(RIF) = {np.mean(RIF):.2f}  (should \u2248 Q_\u03c4 = {Q_tau:.2f})")
print(f"  Std(RIF) = {np.std(RIF):.2f}")
print(f"  Min(RIF) = {np.min(RIF):.2f}, Max(RIF) = {np.max(RIF):.2f}")

# Step 4: Regress RIF on treatment (and constant)
X_rif = np.column_stack([np.ones(n), d_all])
beta_rif = lstsq(X_rif, RIF)[0]

# Compute robust SE
residuals = RIF - X_rif @ beta_rif
meat = X_rif.T @ np.diag(residuals**2) @ X_rif
bread = np.linalg.inv(X_rif.T @ X_rif)
vcov_rif = bread @ meat @ bread / n
se_rif = np.sqrt(np.diag(vcov_rif))

print("\nStep 4: RIF Regression Results:")
print(f"  Intercept: {beta_rif[0]:.2f} (SE: {se_rif[0]:.2f})")
print(f"  Treatment (UQTE): {beta_rif[1]:.2f} (SE: {se_rif[1]:.2f})")
print(f"  t-statistic: {beta_rif[1] / se_rif[1]:.2f}")
print(f"  p-value: {2 * (1 - stats.norm.cdf(abs(beta_rif[1] / se_rif[1]))):.4f}")

# Step 5: Compare with PanelBox
print("\nStep 5: Comparison:")
print(f"  Manual UQTE(0.5): ${beta_rif[1]:.2f}")
try:
    pb_uqte = rif_result.qte_results[0.5]["qte"]
    print(f"  PanelBox UQTE(0.5): ${pb_uqte:.2f}")
    print(f"  Difference: ${abs(beta_rif[1] - pb_uqte):.2f}")
    print(
        f"  => Results {'match closely' if abs(beta_rif[1] - pb_uqte) < 50 else 'differ (PanelBox uses controls)'}!"
    )
except:
    print("  PanelBox result not available for comparison")

print("\nNote: Small differences are expected because PanelBox includes")
print("covariates (education, age, experience) in the RIF regression,")
print("while our manual version only uses the treatment indicator.")

## Exercise 3: Testing Parallel Trends (Medium)

In [ ]:
# SOLUTION: Exercise 3 — Testing Parallel Trends

# Since we only have 2 periods, we'll assess parallel trends by comparing
# the PRE-TREATMENT distributions of treated and control groups.
# Under parallel trends, these distributions should have the same shape.

pre_data = data[data["period"] == 0]
treated_pre = pre_data[pre_data["treatment"] == 1]["earnings"].values
control_pre = pre_data[pre_data["treatment"] == 0]["earnings"].values

post_data = data[data["period"] == 1]
treated_post = post_data[post_data["treatment"] == 1]["earnings"].values
control_post = post_data[post_data["treatment"] == 0]["earnings"].values

tau_test = np.arange(0.05, 0.96, 0.05)

# Compute quantiles for each group-period
q_treated_pre = np.quantile(treated_pre, tau_test)
q_control_pre = np.quantile(control_pre, tau_test)
q_treated_post = np.quantile(treated_post, tau_test)
q_control_post = np.quantile(control_post, tau_test)

# Under parallel trends: Q_\u03c4(Y|D=1,t=0) - Q_\u03c4(Y|D=0,t=0) should be
# approximately equal to Q_\u03c4(Y(0)|D=1,t=1) - Q_\u03c4(Y|D=0,t=1)
# We can't observe the counterfactual, but we can check if the pre-period
# gap is approximately constant across quantiles

pre_gap = q_treated_pre - q_control_pre

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel A: Pre-period quantile comparison
axes[0].plot(
    tau_test, q_treated_pre, "o-", color="#D55E00", label="Treated (Pre)", linewidth=2, markersize=4
)
axes[0].plot(
    tau_test,
    q_control_pre,
    "s-",
    color="steelblue",
    label="Control (Pre)",
    linewidth=2,
    markersize=4,
)
axes[0].set_xlabel("Quantile (\u03c4)", fontsize=12)
axes[0].set_ylabel("Earnings ($)", fontsize=12)
axes[0].set_title("Panel A: Pre-Treatment Quantile Functions", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Panel B: Pre-period gap (should be relatively constant for parallel trends)
axes[1].plot(tau_test, pre_gap, "o-", color="purple", linewidth=2, markersize=5)
axes[1].axhline(
    np.mean(pre_gap),
    color="red",
    linestyle="--",
    linewidth=2,
    label=f"Mean gap = ${np.mean(pre_gap):.0f}",
)
axes[1].fill_between(
    tau_test,
    np.mean(pre_gap) - 2 * np.std(pre_gap),
    np.mean(pre_gap) + 2 * np.std(pre_gap),
    alpha=0.2,
    color="red",
    label="\u00b12 SD band",
)
axes[1].set_xlabel("Quantile (\u03c4)", fontsize=12)
axes[1].set_ylabel("Gap: Treated - Control ($)", fontsize=12)
axes[1].set_title("Panel B: Pre-Treatment Quantile Gap", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

# Panel C: QQ-plot of treated vs control (pre-treatment)
axes[2].scatter(q_control_pre, q_treated_pre, color="purple", s=50, zorder=5)
min_val = min(q_control_pre.min(), q_treated_pre.min())
max_val = max(q_control_pre.max(), q_treated_pre.max())
axes[2].plot([min_val, max_val], [min_val, max_val], "k--", linewidth=1.5, label="45\u00b0 line")
axes[2].set_xlabel("Control Quantiles ($)", fontsize=12)
axes[2].set_ylabel("Treated Quantiles ($)", fontsize=12)
axes[2].set_title("Panel C: QQ-Plot (Pre-Treatment)", fontsize=13, fontweight="bold")
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Formal test: Kolmogorov-Smirnov test on pre-treatment distributions
ks_stat, ks_pval = stats.ks_2samp(treated_pre, control_pre)
print("\nKolmogorov-Smirnov Test (Pre-Treatment):")
print(f"  KS statistic = {ks_stat:.4f}")
print(f"  p-value = {ks_pval:.4f}")
if ks_pval > 0.05:
    print("  => FAIL TO REJECT H0: Pre-treatment distributions are similar")
    print("     This supports (but does not prove) parallel trends")
else:
    print("  => REJECT H0: Pre-treatment distributions differ")
    print("     Parallel trends assumption may be questionable")

# Quantile-by-quantile variability
print("\nPre-treatment gap statistics:")
print(f"  Mean gap: ${np.mean(pre_gap):.0f}")
print(f"  Std of gap: ${np.std(pre_gap):.0f}")
print(f"  CV of gap: {np.std(pre_gap) / max(abs(np.mean(pre_gap)), 1):.2%}")
print(f"  Range: ${pre_gap.min():.0f} to ${pre_gap.max():.0f}")

if np.std(pre_gap) / max(abs(np.mean(pre_gap)), 1) < 0.5:
    print("\n  => Gap is relatively stable across quantiles: parallel trends plausible")
else:
    print("\n  => Gap varies substantially: parallel trends may be questionable")
    print("     DiD-QR results should be interpreted with caution")

## Exercise 4: ATE Hides Heterogeneity (Medium)

In [ ]:
# SOLUTION: Exercise 4 — ATE Hides Heterogeneity

np.random.seed(42)
n_ex4 = 2000

# Generate base earnings (ability-driven)
ability = np.random.normal(0, 1, n_ex4)
base_earnings = 3000 + 800 * ability + np.abs(np.random.normal(0, 500, n_ex4))

# Treatment assignment (random)
treatment_ex4 = np.random.binomial(1, 0.5, n_ex4)

# HETEROGENEOUS treatment effects:
# Low earners (low ability): gain $1000
# High earners (high ability): LOSE $200
# Effect transitions smoothly via sigmoid
te_ex4 = 1000 - 1200 * stats.norm.cdf(ability)
# This gives: TE \u2248 +$1000 at low ability, TE \u2248 -$200 at high ability

# Observed earnings
earnings_ex4 = base_earnings + treatment_ex4 * te_ex4

# Compute ATE
ate_ex4 = np.mean(earnings_ex4[treatment_ex4 == 1]) - np.mean(earnings_ex4[treatment_ex4 == 0])
print(f"AVERAGE TREATMENT EFFECT: ${ate_ex4:.0f}")
print("  => ATE is positive, suggesting the program 'works'")
print("  => But is this the full story?\n")

# Compute QTE
tau_ex4 = np.arange(0.05, 0.96, 0.05)
qte_ex4 = []
for tau in tau_ex4:
    q1 = np.quantile(earnings_ex4[treatment_ex4 == 1], tau)
    q0 = np.quantile(earnings_ex4[treatment_ex4 == 0], tau)
    qte_ex4.append(q1 - q0)

print("QUANTILE TREATMENT EFFECTS:")
print(f"{'\u03c4':<6} {'QTE':>10}")
print("-" * 20)
for tau_val, qte_val in zip(
    [0.1, 0.25, 0.5, 0.75, 0.9], [qte_ex4[1], qte_ex4[4], qte_ex4[9], qte_ex4[14], qte_ex4[17]]
):
    sig = "(+)" if qte_val > 0 else "(-)"
    print(f"{tau_val:<6.2f} ${qte_val:>9.0f} {sig}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: QTE across quantiles
ax1.plot(
    tau_ex4, qte_ex4, "o-", color="steelblue", linewidth=2.5, markersize=5, label="QTE(\u03c4)"
)
ax1.axhline(ate_ex4, color="red", linestyle="--", linewidth=2, label=f"ATE = ${ate_ex4:.0f}")
ax1.axhline(0, color="black", linestyle=":", alpha=0.5, linewidth=1.5)
ax1.fill_between(
    tau_ex4, qte_ex4, 0, where=np.array(qte_ex4) > 0, alpha=0.15, color="green", label="Beneficial"
)
ax1.fill_between(
    tau_ex4, qte_ex4, 0, where=np.array(qte_ex4) < 0, alpha=0.15, color="red", label="Harmful"
)
ax1.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Treatment Effect ($)", fontsize=12, fontweight="bold")
ax1.set_title("QTE Reveals What ATE Hides", fontsize=14, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Panel B: Distribution comparison
bins_ex4 = np.linspace(base_earnings.min() - 500, base_earnings.max() + 1500, 40)
ax2.hist(
    earnings_ex4[treatment_ex4 == 0],
    bins=bins_ex4,
    alpha=0.5,
    density=True,
    color="steelblue",
    label="Control",
)
ax2.hist(
    earnings_ex4[treatment_ex4 == 1],
    bins=bins_ex4,
    alpha=0.5,
    density=True,
    color="#D55E00",
    label="Treated",
)
ax2.axvline(
    np.mean(earnings_ex4[treatment_ex4 == 0]), color="steelblue", linestyle="--", linewidth=2
)
ax2.axvline(np.mean(earnings_ex4[treatment_ex4 == 1]), color="#D55E00", linestyle="--", linewidth=2)
ax2.set_xlabel("Earnings ($)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Density", fontsize=12, fontweight="bold")
ax2.set_title("Earnings Distributions", fontsize=14, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Policy conclusions
print("\nPOLICY CONCLUSIONS:")
print("=" * 60)
print(f"\nWith ATE alone (${ate_ex4:.0f}):")
print("  'The program has a positive average effect \u2014 fund it!'")
print("\nWith QTE analysis:")
print(f"  QTE(0.1) = ${qte_ex4[1]:.0f} \u2014 Low earners benefit substantially")
print(
    f"  QTE(0.5) = ${qte_ex4[9]:.0f} \u2014 Median earners see {'positive' if qte_ex4[9] > 0 else 'mixed'} effects"
)
print(
    f"  QTE(0.9) = ${qte_ex4[17]:.0f} \u2014 High earners are {'hurt' if qte_ex4[17] < 0 else 'unaffected'}"
)
print("\n  => The program HELPS low earners but HURTS high earners!")
print("  => ATE alone masks this crucial distributional impact")
print("  => Better policy: TARGET the program to low earners only")

## Exercise 5: DiD-QR vs Changes-in-Changes (Hard)

In [ ]:
# SOLUTION: Exercise 5 — DiD-QR vs Changes-in-Changes

np.random.seed(42)
n_ex5 = 500

# Simulate data where parallel trends FAILS
# Key: control group's variance changes more than treated group's

# Pre-period (both groups have similar distributions)
y_treated_pre = np.random.normal(5000, 500, n_ex5)
y_control_pre = np.random.normal(5000, 500, n_ex5)

# Post-period:
# Control group: mean increases AND variance increases (e.g., increasing inequality)
y_control_post = np.random.normal(5300, 900, n_ex5)

# Treated group: mean increases + constant TE of $400, variance stays same
true_te = 400  # Constant TE at all quantiles
y_treated_post = np.random.normal(5300, 500, n_ex5) + true_te

# Construct panel DataFrame
ids = np.concatenate([np.arange(1, n_ex5 + 1)] * 2 + [np.arange(n_ex5 + 1, 2 * n_ex5 + 1)] * 2)
periods = np.concatenate([np.zeros(n_ex5), np.ones(n_ex5), np.zeros(n_ex5), np.ones(n_ex5)])
treatments = np.concatenate([np.ones(2 * n_ex5), np.zeros(2 * n_ex5)])
earnings_ex5 = np.concatenate([y_treated_pre, y_treated_post, y_control_pre, y_control_post])

df_ex5 = pd.DataFrame(
    {
        "id": ids.astype(int),
        "period": periods.astype(int),
        "treatment": treatments.astype(int),
        "earnings": earnings_ex5,
    }
)

print(f"Simulated data: {df_ex5.shape}")
print(f"True treatment effect: ${true_te} (constant across quantiles)")
print("\nKey design:")
print(f"  Control group: variance {500} \u2192 {900} (increasing inequality)")
print(f"  Treated group: variance {500} \u2192 {500} (stable)")
print("  => Parallel trends in quantiles VIOLATED at extremes")

# Estimate DiD-QR
qte_did_ex5 = QuantileTreatmentEffects(
    data=df_ex5, outcome="earnings", treatment="treatment", entity_col="id", time_col="period"
)

tau_ex5 = np.array([0.05, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95])

try:
    did_result_ex5 = qte_did_ex5.estimate_qte(tau=tau_ex5, method="did", bootstrap=False)
    did_vals_ex5 = [did_result_ex5.qte_results[tau]["qte"] for tau in tau_ex5]
except Exception as e:
    print(f"DiD error: {e}")
    did_vals_ex5 = [np.nan] * len(tau_ex5)

# Estimate CiC
try:
    cic_result_ex5 = qte_did_ex5.estimate_qte(tau=tau_ex5, method="cic")
    cic_vals_ex5 = [cic_result_ex5.qte_results[tau]["qte"] for tau in tau_ex5]
except Exception as e:
    print(f"CiC error: {e}")
    cic_vals_ex5 = [np.nan] * len(tau_ex5)

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Panel A: Compare methods
ax1.plot(
    tau_ex5, did_vals_ex5, "o-", color="steelblue", linewidth=2.5, markersize=7, label="DiD-QR"
)
ax1.plot(
    tau_ex5,
    cic_vals_ex5,
    "s-",
    color="#D55E00",
    linewidth=2.5,
    markersize=7,
    label="Changes-in-Changes",
)
ax1.axhline(
    true_te, color="green", linestyle="--", linewidth=2.5, label=f"True TE = ${true_te}", alpha=0.8
)
ax1.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax1.set_ylabel("Estimated Treatment Effect ($)", fontsize=12, fontweight="bold")
ax1.set_title("DiD-QR vs CiC\n(Parallel Trends Violated)", fontsize=14, fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(alpha=0.3)

# Panel B: Bias comparison
did_bias = [v - true_te for v in did_vals_ex5]
cic_bias = [v - true_te for v in cic_vals_ex5]
ax2.bar(
    np.array(range(len(tau_ex5))) - 0.15,
    did_bias,
    width=0.3,
    color="steelblue",
    alpha=0.7,
    label="DiD-QR Bias",
)
ax2.bar(
    np.array(range(len(tau_ex5))) + 0.15,
    cic_bias,
    width=0.3,
    color="#D55E00",
    alpha=0.7,
    label="CiC Bias",
)
ax2.axhline(0, color="black", linestyle="-", linewidth=1)
ax2.set_xticks(range(len(tau_ex5)))
ax2.set_xticklabels([f"{t:.2f}" for t in tau_ex5], rotation=45)
ax2.set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
ax2.set_ylabel("Bias (Estimated - True) ($)", fontsize=12, fontweight="bold")
ax2.set_title("Estimation Bias by Method", fontsize=14, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
print("\nSUMMARY:")
print("=" * 65)
print(f"{'\u03c4':<6} {'DiD-QR':>10} {'CiC':>10} {'True':>8} {'DiD Bias':>10} {'CiC Bias':>10}")
print("-" * 65)
for i, tau in enumerate(tau_ex5):
    did_v = did_vals_ex5[i]
    cic_v = cic_vals_ex5[i]
    d_bias = did_v - true_te
    c_bias = cic_v - true_te
    print(f"{tau:<6.2f} {did_v:>10.0f} {cic_v:>10.0f} {true_te:>8} {d_bias:>10.0f} {c_bias:>10.0f}")

did_rmse = np.sqrt(np.nanmean([(v - true_te) ** 2 for v in did_vals_ex5]))
cic_rmse = np.sqrt(np.nanmean([(v - true_te) ** 2 for v in cic_vals_ex5]))
print(f"\nRMSE \u2014 DiD-QR: ${did_rmse:.0f},  CiC: ${cic_rmse:.0f}")
if cic_rmse < did_rmse:
    print("=> CiC is MORE ACCURATE when parallel trends in quantiles fails")
else:
    print("=> Both methods perform similarly in this scenario")

print("\nCONCLUSION:")
print("When the control group's variance changes differently from the treated group,")
print("parallel trends in quantiles is violated. DiD-QR overestimates effects at")
print("extreme quantiles (where the variance change is largest), while CiC adapts")
print("to the distributional changes more robustly.")

## Exercise 6: Minimum Wage Policy Analysis (Hard)

In [ ]:
# SOLUTION: Exercise 6 — Minimum Wage Policy Analysis

np.random.seed(42)
n_ex6 = 2000

# Generate pre-treatment wage distribution (log-normal, realistic)
log_wages = np.random.normal(np.log(3000), 0.5, n_ex6)
base_wages = np.exp(log_wages)

# Treatment: minimum wage increase from $1500 to $1800
old_min_wage = 1500
new_min_wage = 1800

# Simulate who is affected:
# Group 1: Workers below new minimum (get a raise)
# Group 2: Workers near new minimum (possible hours reduction / spillover)
# Group 3: Workers well above minimum (unaffected)

treatment_mw = np.random.binomial(1, 0.5, n_ex6)  # Treatment = region with new MW


# Effect function:
def minimum_wage_effect(wage, new_min=1800, old_min=1500):
    """Simulate minimum wage increase effect on individual wages."""
    if wage < old_min:
        # Below old minimum: brought up to new minimum
        return new_min - wage
    elif wage < new_min:
        # Between old and new minimum: brought up to new minimum
        return new_min - wage
    elif wage < new_min * 1.3:
        # Spillover zone: small positive effect (wage compression pushback)
        spillover = 100 * np.exp(-(wage - new_min) / 200)
        return spillover
    else:
        # Well above minimum: no effect
        return 0


# Apply effects
effects = np.array([minimum_wage_effect(w) for w in base_wages])
post_wages = base_wages.copy()
post_wages[treatment_mw == 1] += effects[treatment_mw == 1]

# Small hours reduction for some low-wage workers in treated group
# (some employers cut hours \u2192 effective wage drop)
hours_reduction = np.random.binomial(1, 0.1, n_ex6)  # 10% lose hours
post_wages[(treatment_mw == 1) & (hours_reduction == 1) & (base_wages < new_min_wage)] *= 0.85

# Compute QTE
tau_mw = np.arange(0.05, 0.96, 0.05)
qte_mw = []
for tau in tau_mw:
    q1 = np.quantile(post_wages[treatment_mw == 1], tau)
    q0 = np.quantile(post_wages[treatment_mw == 0], tau)
    qte_mw.append(q1 - q0)

ate_mw = np.mean(post_wages[treatment_mw == 1]) - np.mean(post_wages[treatment_mw == 0])

# Visualization
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel A: Wage distributions
bins_mw = np.linspace(1000, 8000, 50)
axes[0].hist(
    post_wages[treatment_mw == 0],
    bins=bins_mw,
    alpha=0.5,
    density=True,
    color="steelblue",
    label="No MW increase",
)
axes[0].hist(
    post_wages[treatment_mw == 1],
    bins=bins_mw,
    alpha=0.5,
    density=True,
    color="#D55E00",
    label="MW increase",
)
axes[0].axvline(
    old_min_wage, color="gray", linestyle=":", linewidth=2, label=f"Old MW = ${old_min_wage}"
)
axes[0].axvline(
    new_min_wage, color="green", linestyle="--", linewidth=2, label=f"New MW = ${new_min_wage}"
)
axes[0].set_xlabel("Monthly Wage ($)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Density", fontsize=12, fontweight="bold")
axes[0].set_title("Panel A: Wage Distributions", fontsize=13, fontweight="bold")
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Panel B: QTE across distribution
axes[1].plot(tau_mw, qte_mw, "o-", color="purple", linewidth=2.5, markersize=5, label="QTE(\u03c4)")
axes[1].axhline(ate_mw, color="red", linestyle="--", linewidth=2, label=f"ATE = ${ate_mw:.0f}")
axes[1].axhline(0, color="black", linestyle=":", alpha=0.5)
axes[1].fill_between(tau_mw, qte_mw, 0, where=np.array(qte_mw) > 0, alpha=0.15, color="green")
axes[1].fill_between(tau_mw, qte_mw, 0, where=np.array(qte_mw) < 0, alpha=0.15, color="red")
axes[1].set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
axes[1].set_ylabel("Treatment Effect ($)", fontsize=12, fontweight="bold")
axes[1].set_title("Panel B: QTE of Minimum Wage Increase", fontsize=13, fontweight="bold")
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)
axes[1].annotate(
    "MW directly\nraises wages",
    xy=(0.1, qte_mw[1]),
    fontsize=9,
    xytext=(0.25, qte_mw[1] + 50),
    arrowprops={"arrowstyle": "->", "color": "green"},
)
axes[1].annotate(
    "No effect on\nhigh earners",
    xy=(0.85, qte_mw[-3]),
    fontsize=9,
    xytext=(0.65, qte_mw[-3] + 100),
    arrowprops={"arrowstyle": "->", "color": "gray"},
)

# Panel C: Cumulative effect (who benefits?)
pct_positive = [np.mean(np.array(qte_mw[: i + 1]) > 0) * 100 for i in range(len(tau_mw))]
axes[2].plot(
    tau_mw,
    np.cumsum(qte_mw) / np.arange(1, len(tau_mw) + 1),
    "o-",
    color="#009E73",
    linewidth=2.5,
    markersize=5,
)
axes[2].axhline(0, color="black", linestyle=":", alpha=0.5)
axes[2].set_xlabel("Quantile (\u03c4)", fontsize=12, fontweight="bold")
axes[2].set_ylabel("Cumulative Average QTE ($)", fontsize=12, fontweight="bold")
axes[2].set_title("Panel C: Cumulative Average Effect", fontsize=13, fontweight="bold")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# Policy analysis
print("\nPOLICY ANALYSIS: Minimum Wage Increase ($1500 \u2192 $1800)")
print("=" * 60)
print(f"\nATE = ${ate_mw:.0f}")
print(
    f"  => With ATE alone: 'Minimum wage increase {'raises' if ate_mw > 0 else 'reduces'} average wages'"
)

print("\nQTE Analysis:")
print(
    f"  QTE(0.05) = ${qte_mw[0]:.0f} \u2014 Lowest earners: {'benefit' if qte_mw[0] > 0 else 'may be hurt'}"
)
print(f"  QTE(0.10) = ${qte_mw[1]:.0f} \u2014 Low earners: large positive effect")
print(f"  QTE(0.25) = ${qte_mw[4]:.0f} \u2014 Below-median workers")
print(
    f"  QTE(0.50) = ${qte_mw[9]:.0f} \u2014 Median workers: {'moderate' if abs(qte_mw[9]) < 200 else 'substantial'} effect"
)
print(
    f"  QTE(0.75) = ${qte_mw[14]:.0f} \u2014 Above-median: {'minimal' if abs(qte_mw[14]) < 50 else 'some'} effect"
)
print(f"  QTE(0.90) = ${qte_mw[17]:.0f} \u2014 High earners: negligible")

print("\nPolicy Conclusions:")
print("  1. The MW increase effectively raises wages at the bottom of the distribution")
print("  2. Effects are concentrated in the lower 30% of the wage distribution")
print("  3. High earners are essentially unaffected (QTE \u2192 0)")
print("  4. This is a PROGRESSIVE policy (QTE strongly decreasing in \u03c4)")
print(f"  5. ATE of ${ate_mw:.0f} understates the benefit for low earners")
print("     and overstates the impact on the overall labor market")

---

## Summary

All exercises demonstrate key aspects of Quantile Treatment Effects:

1. **Ex 1**: CQTE vs UQTE — conditional vs unconditional effects and policy relevance
2. **Ex 2**: Manual RIF — understanding the mechanics behind unconditional QTE
3. **Ex 3**: Parallel trends — validating assumptions for DiD-QR
4. **Ex 4**: ATE masks heterogeneity — the fundamental motivation for QTE
5. **Ex 5**: DiD-QR vs CiC — robustness to assumption violations
6. **Ex 6**: Minimum wage analysis — real-world policy application

**Key takeaway**: QTE provides a much richer picture of treatment effects than ATE alone, and is essential for informed policy design.